# Robust Relevance-Pursuit GP: End-to-End BO with Outliers

少数の強い外れ値を含む観測に対して`RobustRelevancePursuitSingleTaskGP`を学習し、posterior samplingからqLogEI、candidate generation、評価、updateまで接続します。

関連理論:

- [Robust GP](../../docs/theory/14_robust_gaussian_process.md)
- [Bayesian Optimization](../../docs/theory/01_bayesian_optimization.md)

```text
corrupted observations
        ↓
robust relevance-pursuit GP
        ↓
posterior / sampling
        ↓
qLogExpectedImprovement
        ↓
optimize_acqf
        ↓
candidate → evaluation → update
```

ここでrobustなのは主に**観測外れ値に対するsurrogate fitting**です。入力摂動に対するrobust objectiveやworst-case optimizationとは別概念です。

## このNotebookで学ぶこと

- 通常のGPが**少数の大きな観測外れ値**に影響される理由
- Relevance Pursuitによるrobust surrogateの位置付け
- 通常GPとの事後平均・不確実性の比較
- qLogEIを用いた候補提案、評価、観測追加、再学習
- **外れ値に頑健なモデル**と**頑健な獲得関数・目的関数**の違い

このNotebookでは入力は1次元、出力は連続値です。まず通常GPとの比較を行い、その後にBoTorchの標準的な最適化フローへ接続します。

## 1. Imports and reproducibility

In [ ]:
import matplotlib.pyplot as plt
import torch
from botorch.acquisition.logei import qLogExpectedImprovement
from botorch.fit import fit_gpytorch_mll
from botorch.optim import optimize_acqf
from botorch.sampling.normal import SobolQMCNormalSampler

from robotorchan.models import (
    RobustRelevancePursuitSingleTaskGP,
    SingleTaskGP,
)

torch.set_default_dtype(torch.double)
torch.manual_seed(0)

### 外れ値がGPに与える影響

通常のGP回帰は、指定した尤度とノイズ仮定の下で観測値に適合します。少数の観測だけが大きく破損すると、平均予測が局所的に引っ張られ、最大観測値も実際の目的関数の最大値から大きくずれる可能性があります。

この例では28点中3点に人工的な外れ値を加えます。`clean_Y` は合成関数の真値、`train_Y` はノイズと外れ値を含む観測値です。実際の問題では真値は通常わかりません。

## 2. Objective and corrupted observations

latent objectiveは滑らかですが、training observationsの一部だけを強く破損させます。candidateを評価するときはlatent objectiveを使い、新しい外れ値を意図的には追加しません。

In [ ]:
bounds = torch.tensor([[0.0], [1.0]])


def objective(X: torch.Tensor) -> torch.Tensor:
    return torch.sin(2 * torch.pi * X) + 0.25 * X


train_X = torch.linspace(0.0, 1.0, 28).unsqueeze(-1)
clean_Y = objective(train_X)
train_Y = clean_Y + 0.04 * torch.randn_like(clean_Y)
outlier_idx = torch.tensor([5, 17, 23])
train_Y[outlier_idx] += torch.tensor([[1.4], [-1.6], [1.2]])
print(train_X.shape, train_Y.shape)

### 学習データと外れ値の確認

`outlier_idx=[5,17,23]` にだけ大きな加算誤差を入れています。これは**外れ値の発生位置が既知の合成例**であり、実データで外れ値ラベルが利用できることを仮定するものではありません。

robust surrogateの性能を比較するため、通常GPとrobust GPの双方に**同じ破損データ**を渡します。

## 3. Baseline and robust surrogate

同じcorrupted dataへ通常GPとrobust relevance-pursuit GPをfitします。robust modelもexact-GPの`make_mll()` contractを維持します。

In [ ]:
baseline = SingleTaskGP(train_X, train_Y)
fit_gpytorch_mll(baseline.make_mll())
baseline.eval()

robust = RobustRelevancePursuitSingleTaskGP(
    train_X,
    train_Y,
    convex_parameterization=True,
)
fit_gpytorch_mll(robust.make_mll())
robust.eval()

print("raw data:", robust.raw_train_X.shape, robust.raw_train_Y.shape)
print("supports_mll:", robust.supports_mll)

### Relevance Pursuitの考え方

Relevance Pursuitは、通常のGPが説明しづらい少数の観測に対して、外れ値に対応する成分を推定するアプローチです。通常のGPより外れ値の影響を抑えられる場合がありますが、外れ値の割合・大きさ・モデル仮定によって結果は変わります。

`convex_parameterization=True` はこのモデルの推定方法に関する設定です。**頑健性が必ず改善することを保証するスイッチではありません**。

## 4. Posterior comparison

In [ ]:
test_X = torch.linspace(0.0, 1.0, 200).unsqueeze(-1)
with torch.no_grad():
    baseline_posterior = baseline.posterior(test_X)
    robust_posterior = robust.posterior(test_X)

baseline_mean = baseline_posterior.mean.squeeze(-1)
robust_mean = robust_posterior.mean.squeeze(-1)
robust_std = robust_posterior.variance.clamp_min(0).sqrt().squeeze(-1)

fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(train_X.squeeze(-1), train_Y.squeeze(-1), label="observations")
ax.scatter(
    train_X[outlier_idx].squeeze(-1),
    train_Y[outlier_idx].squeeze(-1),
    marker="x",
    s=80,
    label="corrupted observations",
)
ax.plot(test_X.squeeze(-1), baseline_mean, label="SingleTaskGP")
ax.plot(test_X.squeeze(-1), robust_mean, label="Robust GP")
ax.fill_between(
    test_X.squeeze(-1),
    robust_mean - 2 * robust_std,
    robust_mean + 2 * robust_std,
    alpha=0.2,
)
ax.legend()
ax.set_xlabel("x")
ax.set_ylabel("y")
plt.show()

### 予測平均と不確実性の比較

外れ値を含む観測点と、通常GP・robust GPの事後平均を比較します。塗りつぶしはrobust GPの `posterior.variance` から計算した標準偏差の約2倍であり、**潜在関数の近似事後区間**です。観測ノイズ込みの予測区間ではありません。

予測曲線の見た目だけでなく、合成関数の真値がわかるこの例ではRMSEも計算できます。

In [ ]:
with torch.no_grad():
    true_test_Y = objective(test_X).squeeze(-1)
    baseline_rmse = torch.sqrt(torch.mean((baseline_mean - true_test_Y) ** 2))
    robust_rmse = torch.sqrt(torch.mean((robust_mean - true_test_Y) ** 2))
print("SingleTaskGP latent-function RMSE:", float(baseline_rmse))
print("Robust GP latent-function RMSE:", float(robust_rmse))
print("This metric is specific to the synthetic test grid, not a general guarantee.")

## 5. Posterior sampling

Relevance-pursuit modelはstandard Gaussian posterior sampling contractを維持するため、BoTorchのQMC samplerをそのまま利用できます。

In [ ]:
sample_X = torch.tensor([[0.25], [0.75]])
sampler = SobolQMCNormalSampler(sample_shape=torch.Size([64]), seed=123)
samples = sampler(robust.posterior(sample_X))

assert samples.shape == torch.Size([64, 2, 1])
assert torch.isfinite(samples).all()
print("samples:", samples.shape)

### Posterior samplingと獲得関数の関係

`SobolQMCNormalSampler` は事後分布からサンプルを生成します。qLogEIは候補の改善量をMonte Carloサンプリングで近似するため、モデルがBoTorchの標準posterior APIを満たすことが重要です。

ここでは2点で64サンプルを生成し、形状 `(64, 2, 1)` を確認します。これは**モデルの事後サンプリング**であり、観測データに外れ値を追加する操作ではありません。

## 6. Acquisition function

`best_f`には観測最大値をそのまま使います。この簡潔な例ではrobust incumbent estimatorまでは導入しません。そのため、極端な正の外れ値が`best_f`を押し上げ得る点は重要なcaveatです。

In [ ]:
acqf = qLogExpectedImprovement(
    model=robust,
    best_f=train_Y.max(),
    sampler=SobolQMCNormalSampler(sample_shape=torch.Size([128]), seed=456),
)
probe_X = torch.tensor([[[0.4]]])
print("qLogEI probe:", acqf(probe_X))

### `best_f` の注意点：robust GPでも外れ値は残る

EIの改善基準 `best_f` は、どの値を超えたら改善とみなすかを決めます。この例では `train_Y.max()` を使いますが、**正の外れ値が最大観測値を押し上げる**ため、robust surrogateを使っていても改善基準が不適切になる場合があります。

このNotebookではAPIの接続を示すため既存コードを維持します。実務では潜在関数の推定に基づくincumbentや、観測ノイズを扱うNEI系の方法などを問題設定に応じて検討します。これらは単純な置換で同等になるものではありません。

## 7. Candidate generation

surrogateがrobustでもcandidate optimization APIは通常のBoTorch pathを維持します。

In [ ]:
candidate, acq_value = optimize_acqf(
    acq_function=acqf,
    bounds=bounds,
    q=1,
    num_restarts=5,
    raw_samples=64,
)

assert candidate.shape == torch.Size([1, 1])
assert torch.all(candidate >= bounds[0])
assert torch.all(candidate <= bounds[1])
print("candidate:", candidate)
print("acquisition value:", acq_value)

### 候補提案の読み方

`optimize_acqf` はqLogEIを最大化する候補を探索します。先のNotebookで使った**有限候補集合の事後平均ランキング**とは異なり、事後不確実性も改善可能性の計算に入ります。

`q=1` は1回に1点提案する設定です。`num_restarts` と `raw_samples` は獲得関数最適化の初期化・探索設定で、GPの観測点数ではありません。

## 8. Candidate evaluation and update

In [ ]:
candidate_Y = objective(candidate)
updated_X = torch.cat([train_X, candidate], dim=0)
updated_Y = torch.cat([train_Y, candidate_Y], dim=0)

print("candidate objective:", candidate_Y)
print("updated shapes:", updated_X.shape, updated_Y.shape)

### 候補の評価とデータ更新

`candidate` は**次に評価すべき入力**、`candidate_Y` は合成目的関数による評価結果です。観測を追加した後は学習データが28点から29点になります。

実際の実験では評価値に測定ノイズや外れ値が含まれる可能性があります。この例は理解を容易にするため、新規候補の評価には外れ値を追加していません。

## 9. Refit and generate the next candidate

In [ ]:
next_model = RobustRelevancePursuitSingleTaskGP(
    updated_X,
    updated_Y,
    convex_parameterization=True,
)
fit_gpytorch_mll(next_model.make_mll())
next_model.eval()
next_acqf = qLogExpectedImprovement(
    model=next_model,
    best_f=updated_Y.max(),
    sampler=SobolQMCNormalSampler(sample_shape=torch.Size([128]), seed=789),
)
next_candidate, _ = optimize_acqf(
    acq_function=next_acqf,
    bounds=bounds,
    q=1,
    num_restarts=5,
    raw_samples=64,
)
print("next candidate:", next_candidate)

### 再学習して次の候補を提案する

新しい観測を追加したら、更新データでrobust GPを再学習し、獲得関数を作り直します。これが**モデル更新 → 獲得関数更新 → 次候補提案**というBOループの基本形です。

この例では `best_f=updated_Y.max()` を再利用しているため、最初の外れ値が改善基準に影響し続ける可能性があります。モデルがrobustでも、最適化ループ全体の頑健性が保証されるわけではありません。

## 10. Robustness layers are different

| Failure mode | Representative model / mechanism | Meaning |
| --- | --- | --- |
| sparse gross outliers | Relevance Pursuit | observation robustness |
| heavy-tailed residuals | Student-t | robust likelihood |
| explicit contamination | Contaminated GP | mixture observation model |
| repeated-measurement noise | ReplicateNoise GP | heteroskedastic observation variance |
| uncertain continuous X | UncertainInput GP | input uncertainty |
| uncertain category | UncertainCategorical GP | category-probability marginalization |
| varying smoothness | Nonstationary GP | latent-function structure |
| DKL / DeepGP / spectral kernels | expressive surrogates | representation / covariance expressivity |

これらはすべて「robust GP」という同一問題ではありません。

## 11. Surrogate robustness is not robust optimization

```text
Robust surrogate
    corrupted/noisy training dataへのmodeling response

Robust objective / input perturbation
    candidate xの周辺や環境変動を含めてutilityを定義

Robust acquisition / risk measure
    expectation, variance, VaR, CVaR等でdecision riskを評価
```

このNotebookが直接実証するのは1つ目です。入力摂動を含むrobust BO workflowは後続Phaseで扱います。

## 12. When to use and caveats

- 少数の観測が破損し得る場合にrelevance pursuitを検討します。
- 外れ値がheavy-tailやcontamination process由来なら専用observation modelも比較します。
- 外れ値に見える点が別regimeならmulti-task / conditional / nonstationary modelの方が自然な場合があります。
- robust surrogateを使っても`best_f`やobjective definitionまで自動的にrobustにはなりません。
- 表現力の高いmodelはrobustnessの代替ではありません。DKLやDeepGPは別のmodeling assumptionです。
- standard GPとの予測・校正・BO performance比較を行って採用します。

## 13. Related notebooks

- `15_robust_observation_models.ipynb`: Student-t / contamination
- `16_noise_models.ipynb`: replicate-based observation noise
- `17_uncertain_input_gp.ipynb`: continuous / categorical input uncertainty
- `18_nonstationary_gp.ipynb`: input-dependent smoothness
- `24_expressive_surrogate_gp.ipynb`: DKL / I-BNN / Spectral Mixture / DeepGP

各Notebookは異なるmodeling semanticsを保持し、このNotebookへ無理に統合しません。

### MixedとMultiTaskへの拡張

最後の例では、カテゴリ設計変数を含むMixed GPと、複数タスクの情報を共有するMultiTask GPを扱います。

- **Mixed**：候補のカテゴリ値を列挙し、連続変数と一緒に最適化します
- **MultiTask**：タスクIDで関連する応答を区別し、タスク間の情報共有を利用します
- **Robust**：学習データに含まれる観測外れ値への対処です

これらは互いに異なる役割を持ちます。カテゴリ変数とtask IDを混同しないようにしてください。

## 14. Mixed単一タスクとRobust MultiTaskの候補提案

Mixed単一タスクに加え、通常型・Mixed型のRobust Relevance Pursuit MultiTask GPを実際に学習し、task 0に対するqLogEI候補提案と評価を行います。


In [ ]:
from botorch.acquisition.objective import ScalarizedPosteriorTransform
from botorch.optim.optimize import optimize_acqf_mixed

from robotorchan.models import (
    MixedRobustRelevancePursuitMultiTaskGP,
    MixedRobustRelevancePursuitSingleTaskGP,
    RobustRelevancePursuitMultiTaskGP,
)

mixed_X = torch.cat(
    [train_X, (torch.arange(len(train_X)) % 2).to(train_X).unsqueeze(-1)],
    dim=-1,
)

def mixed_objective(X):
    return objective(X[:, :1]) + 0.15 * X[:, 1:2]


mixed_Y = mixed_objective(mixed_X) + 0.03 * torch.randn(len(mixed_X), 1)
mixed_Y[[5, 17, 23]] += torch.tensor([[1.4], [-1.6], [1.2]])
mixed_robust = MixedRobustRelevancePursuitSingleTaskGP(
    mixed_X, mixed_Y, cat_dims=[1], convex_parameterization=True,
)
fit_gpytorch_mll(mixed_robust.make_mll())
mixed_robust.eval()

mixed_acqf = qLogExpectedImprovement(
    model=mixed_robust,
    best_f=mixed_Y.max(),
    sampler=SobolQMCNormalSampler(sample_shape=torch.Size([64]), seed=301),
)
mixed_bounds = torch.tensor([[0.0, 0.0], [1.0, 1.0]])
mixed_candidate, _ = optimize_acqf_mixed(
    acq_function=mixed_acqf,
    bounds=mixed_bounds,
    q=1,
    num_restarts=3,
    raw_samples=32,
    fixed_features_list=[{1: 0.0}, {1: 1.0}],
)
assert mixed_candidate[0, 1].item() in (0.0, 1.0)
print("Mixed single-task:", mixed_candidate, mixed_objective(mixed_candidate))

# Long-format observations: the final column is the task index.
for mixed in (False, True):
    design = torch.linspace(0.05, 0.95, 12).unsqueeze(-1)
    if mixed:
        category = (torch.arange(12) % 2).to(design).unsqueeze(-1)
        design = torch.cat([design, category], dim=-1)
    task0 = torch.zeros(12, 1, dtype=design.dtype)
    task1 = torch.ones(12, 1, dtype=design.dtype)
    multitask_X = torch.cat(
        [torch.cat([design, task0], dim=-1),
         torch.cat([design, task1], dim=-1)]
    )
    values = objective(design[:, :1])
    if mixed:
        values = values + 0.15 * design[:, 1:2]
    multitask_Y = torch.cat([values, 0.8 * values + 0.1], dim=0)
    multitask_Y[3] += 1.0
    model_cls = (
        MixedRobustRelevancePursuitMultiTaskGP
        if mixed else RobustRelevancePursuitMultiTaskGP
    )
    model_kwargs = {
        "task_feature": -1,
        "output_tasks": [0],
        "convex_parameterization": True,
    }
    if mixed:
        model_kwargs["cat_dims"] = [1]
    multitask_model = model_cls(multitask_X, multitask_Y, **model_kwargs)
    fit_gpytorch_mll(multitask_model.make_mll())
    multitask_model.eval()
    task0_X = multitask_X[:12]
    multitask_posterior = multitask_model.posterior(task0_X)
    assert torch.isfinite(multitask_posterior.mean).all()

    task0_best = multitask_Y[:12].max()
    multitask_acqf = qLogExpectedImprovement(
        model=multitask_model,
        best_f=task0_best,
        sampler=SobolQMCNormalSampler(sample_shape=torch.Size([32]), seed=411),
    )
    if mixed:
        multitask_bounds = torch.tensor([[0., 0., 0.], [1., 1., 0.]])
        multitask_candidate, _ = optimize_acqf_mixed(
            acq_function=multitask_acqf,
            bounds=multitask_bounds,
            q=1,
            num_restarts=2,
            raw_samples=16,
            fixed_features_list=[{1: 0., 2: 0.}, {1: 1., 2: 0.}],
        )
    else:
        multitask_bounds = torch.tensor([[0., 0.], [1., 0.]])
        multitask_candidate, _ = optimize_acqf(
            acq_function=multitask_acqf,
            bounds=multitask_bounds,
            q=1,
            num_restarts=2,
            raw_samples=16,
            fixed_features={1: 0.},
        )
    candidate_value = objective(multitask_candidate[:, :1])
    if mixed:
        candidate_value += 0.15 * multitask_candidate[:, 1:2]
    assert torch.isfinite(candidate_value).all()
    print(model_cls.__name__, multitask_candidate, candidate_value)
